In [1]:
print("=" * 80)
print("SILVER LEARNER PROFILES")
print("=" * 80)

spark.table(
    "demo.silver.learner_profiles"
).printSchema()

spark.table(
    "demo.silver.learner_profiles"
).show(truncate=False)


print("=" * 80)
print("GOLD DIM LEARNER")
print("=" * 80)

spark.table(
    "demo.gold.dim_learner"
).printSchema()

spark.table(
    "demo.gold.dim_learner"
).show(truncate=False)

SILVER LEARNER PROFILES
root
 |-- user_id: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- background_level: string (nullable = true)
 |-- learning_goal: string (nullable = true)
 |-- main_domain: string (nullable = true)
 |-- profile_updated_at: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)



+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_id |registration_date|preferred_language|background_level|learning_goal                                            |main_domain     |profile_updated_at |ingestion_time     |is_current|
+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_001|2026-07-01       |English           |Intermediate    |Strengthen virtual memory and operating systems knowledge|Computer Science|2026-07-23 18:10:00|2026-07-23 18:15:00|true      |
|user_001|2026-07-01       |English           |Beginner        |Improve understanding of operating systems               |Computer Science|2026-07-01 08:00:00|2026-07-01 08:05:00|false     |
|user_002|2026-07-03       |English          

In [2]:
# ============================================================
# DIM_LEARNER - SCD TYPE 2 IMPLEMENTATION
# ============================================================

from pyspark.sql import Window
from pyspark.sql.functions import (
    col,
    lead,
    row_number,
    lit
)

DIM_TABLE = "demo.gold.dim_learner"
SILVER_TABLE = "demo.silver.learner_profiles"


# ------------------------------------------------------------
# 1. Add SCD Type 2 columns only if they do not exist
# ------------------------------------------------------------

existing_columns = spark.table(DIM_TABLE).columns

if "valid_from" not in existing_columns:
    spark.sql(f"""
        ALTER TABLE {DIM_TABLE}
        ADD COLUMN valid_from TIMESTAMP
    """)

if "valid_to" not in existing_columns:
    spark.sql(f"""
        ALTER TABLE {DIM_TABLE}
        ADD COLUMN valid_to TIMESTAMP
    """)

if "is_current" not in existing_columns:
    spark.sql(f"""
        ALTER TABLE {DIM_TABLE}
        ADD COLUMN is_current BOOLEAN
    """)

print("SCD2 columns are available.")


# ------------------------------------------------------------
# 2. Prepare all learner-profile versions from Silver
#
# valid_from = profile_updated_at
# valid_to   = next version's profile_updated_at
# is_current = latest version
# ------------------------------------------------------------

version_window = Window.partitionBy(
    "user_id"
).orderBy(
    "profile_updated_at"
)

profile_versions_df = (
    spark.table(SILVER_TABLE)
    .withColumn(
        "valid_from",
        col("profile_updated_at")
    )
    .withColumn(
        "valid_to",
        lead("profile_updated_at").over(version_window)
    )
    .select(
        "user_id",
        "registration_date",
        "preferred_language",
        "background_level",
        "learning_goal",
        "main_domain",
        "profile_updated_at",
        "valid_from",
        "valid_to",
        "is_current"
    )
)

profile_versions_df.createOrReplaceTempView(
    "learner_profile_versions"
)


# ------------------------------------------------------------
# 3. Update existing Gold rows
#
# Existing rows are the current learner versions.
# Their user_key values remain unchanged.
# ------------------------------------------------------------

spark.sql(f"""
    MERGE INTO {DIM_TABLE} AS target

    USING learner_profile_versions AS source

    ON target.user_id = source.user_id
       AND target.profile_updated_at = source.profile_updated_at

    WHEN MATCHED THEN UPDATE SET
        target.registration_date = source.registration_date,
        target.preferred_language = source.preferred_language,
        target.background_level = source.background_level,
        target.learning_goal = source.learning_goal,
        target.main_domain = source.main_domain,
        target.is_active = true,
        target.valid_from = source.valid_from,
        target.valid_to = source.valid_to,
        target.is_current = source.is_current
""")

print("Existing Gold learner versions were updated.")


# ------------------------------------------------------------
# 4. Find Silver historical versions missing from Gold
# ------------------------------------------------------------

missing_versions_df = spark.sql(f"""
    SELECT source.*
    FROM learner_profile_versions AS source

    LEFT ANTI JOIN {DIM_TABLE} AS target
        ON target.user_id = source.user_id
       AND target.profile_updated_at = source.profile_updated_at
""")


# ------------------------------------------------------------
# 5. Generate new surrogate keys only for historical versions
# ------------------------------------------------------------

current_max_key = (
    spark.table(DIM_TABLE)
    .selectExpr(
        "COALESCE(MAX(user_key), 0) AS max_key"
    )
    .first()["max_key"]
)

new_key_window = Window.orderBy(
    "user_id",
    "profile_updated_at"
)

missing_versions_with_keys_df = (
    missing_versions_df
    .withColumn(
        "user_key",
        row_number().over(new_key_window)
        + lit(current_max_key)
    )
    .withColumn(
        "is_active",
        lit(True)
    )
    .select(
        "user_key",
        "user_id",
        "registration_date",
        "preferred_language",
        "background_level",
        "learning_goal",
        "main_domain",
        "profile_updated_at",
        "is_active",
        "valid_from",
        "valid_to",
        "is_current"
    )
)

missing_count = missing_versions_with_keys_df.count()

if missing_count > 0:
    (
        missing_versions_with_keys_df
        .writeTo(DIM_TABLE)
        .append()
    )

    print(
        f"Inserted {missing_count} historical learner version(s)."
    )
else:
    print("No missing historical versions were found.")


# ------------------------------------------------------------
# 6. Verify the final SCD Type 2 dimension
# ------------------------------------------------------------

print("\nFinal dim_learner SCD Type 2:")

spark.table(DIM_TABLE).select(
    "user_key",
    "user_id",
    "background_level",
    "learning_goal",
    "profile_updated_at",
    "valid_from",
    "valid_to",
    "is_current"
).orderBy(
    "user_id",
    "valid_from"
).show(
    truncate=False
)


# ------------------------------------------------------------
# 7. SCD2 validation checks
# ------------------------------------------------------------

duplicate_current_df = spark.sql(f"""
    SELECT
        user_id,
        COUNT(*) AS current_versions
    FROM {DIM_TABLE}
    WHERE is_current = true
    GROUP BY user_id
    HAVING COUNT(*) > 1
""")

invalid_closed_versions_df = spark.sql(f"""
    SELECT *
    FROM {DIM_TABLE}
    WHERE is_current = false
      AND valid_to IS NULL
""")

duplicate_current_count = duplicate_current_df.count()
invalid_closed_count = invalid_closed_versions_df.count()

print("=" * 90)
print("SCD TYPE 2 VALIDATION")
print("=" * 90)

if duplicate_current_count == 0:
    print("One current version per learner: PASS")
else:
    print("One current version per learner: FAIL")
    duplicate_current_df.show(truncate=False)

if invalid_closed_count == 0:
    print("Historical versions have valid_to: PASS")
else:
    print("Historical versions have valid_to: FAIL")
    invalid_closed_versions_df.show(truncate=False)

print("=" * 90)

SCD2 columns are available.
Existing Gold learner versions were updated.


26/07/29 10:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/29 10:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/29 10:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/29 10:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/29 10:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/29 10:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/29 1

Inserted 1 historical learner version(s).

Final dim_learner SCD Type 2:
+--------+--------+----------------+---------------------------------------------------------+-------------------+-------------------+-------------------+----------+
|user_key|user_id |background_level|learning_goal                                            |profile_updated_at |valid_from         |valid_to           |is_current|
+--------+--------+----------------+---------------------------------------------------------+-------------------+-------------------+-------------------+----------+
|4       |user_001|Beginner        |Improve understanding of operating systems               |2026-07-01 08:00:00|2026-07-01 08:00:00|2026-07-23 18:10:00|false     |
|1       |user_001|Intermediate    |Strengthen virtual memory and operating systems knowledge|2026-07-23 18:10:00|2026-07-23 18:10:00|NULL               |true      |
|2       |user_002|Intermediate    |Improve programming fundamentals and recursion           |202